In [28]:
import os 
from dotenv import load_dotenv
from deepagents import FilesystemPermission, create_deep_agent
from langchain_openai import ChatOpenAI
from tavily import TavilyClient
from typing import Literal
from deepagents.backends import StateBackend,FilesystemBackend
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper

In [2]:
load_dotenv(override=True)
tavily = TavilyClient(os.getenv('TAVILY_API_KEY'))

In [3]:
print (type(tavily))

<class 'tavily.client.TavilyClient'>


In [4]:
def internet_search(
    query:str,
    max_result:int=5,
    topics:Literal["general","news","AI"] = "general",
    include_raw_content:bool=False,
):
    """run a web search"""
    return tavily.search(
        query,
        max_result=max_result,
        include_raw_content=include_raw_content,
        topics= topics,
    )

In [5]:
wikipedia = WikipediaAPIWrapper()
wiki_tool = WikipediaQueryRun(api_wrapper = wikipedia)

In [6]:
tools = [internet_search]

In [7]:
llm=ChatOpenAI(
    api_key=os.getenv('DEEPSEEK_API_KEY'),
    model = 'deepseek-v4-flash',
    base_url='https://api.deepseek.com'
)

In [8]:
research_instructions = """You are an expert researcher. Your job is to conduct thorough research and then write a polished report.

You have access to an internet search tool as your primary means of gathering information.

## `internet_search`

Use this to run an internet search for a given query. You can specify the max number of results to return, the topic, 
and whether raw content should be included.
"""

In [9]:
research_agent =create_deep_agent(
    model = llm,
    tools=tools,
    system_prompt=research_instructions,
) 

In [10]:
evaluator = create_deep_agent(
    model = llm,
    system_prompt= "you are required to evaluate the taks peformed by the agents and check whether they have met the required criteria or not"
)

In [30]:
from langchain.tools import tool
@tool
def remove_file(path: str) -> str:
    """Delete a file from the filesystem."""
    return f"Deleted {path}"


@tool
def fetch_file(path: str) -> str:
    """Read a file from the filesystem."""
    return f"Contents of {path}"


In [ ]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()
config = {"configurable": {"thread_id":"user-1"}}

In [37]:
orchestrator = create_deep_agent(
    model=llm,
    tools=tools,
    backend=FilesystemBackend(root_dir='/home/abhishek/Documents/projects/course/DeepAgent/',virtual_mode=True),
    system_prompt="You are the orchestrator. Choose the required agent if needed to complete tasks or sub-tasks.",
    permissions=[
        FilesystemPermission(
            operations=["write", "read"],
            paths=["/**"],
            mode="allow",
        )
    ],
    interrupt_on={
        "remove_file": True,  # Default: approve, edit, reject, respond
        "notify_email": {"allowed_decisions": ["approve", "reject"]},  # No editing
    },
    checkpointer=memory,
    # memory=["AGENTS.md", "~/.deepagents/preferences.md"],
    skills=["/skills/research/", "skills/web-search/"],
    subagents=[
        {
            "name": "research_agent",
            "description": "Use this agent for research tasks and gathering information.",
            "system_prompt": research_instructions,
            "tools": tools,
        },
        {
            "name": "evaluator",
            "description": "Use this agent to evaluate whether the output meets the required criteria.",
            "system_prompt": "You are required to evaluate the task performed by the agents and check whether they met the required criteria or not.",
            "tools": [],
        },
    ],
)

result = orchestrator.invoke({
    "messages": [
        {"role": "user", "content": "research on latest AI agentic frameworks and how to work on langgraph agents and store it in langgraph.txt file"
            "Then tell me you've done it."}
            
    ]
},config=config,)
print("After invoke")
print(result)
print(result["messages"][-1].content)

Cannot load skills from '/skills/research/': Path '/skills/research/': path_not_found
Cannot load skills from 'skills/web-search/': Path 'skills/web-search/': path_not_found
Skills load errors: ["Cannot load skills from '/skills/research/': Path '/skills/research/': path_not_found", "Cannot load skills from 'skills/web-search/': Path 'skills/web-search/': path_not_found"]


After invoke
{'messages': [HumanMessage(content="research on latest AI agentic frameworks and how to work on langgraph agents and store it in langgraph.txt fileThen tell me you've done it.", additional_kwargs={}, response_metadata={}, id='4f143b55-2fab-4156-8dda-30c8355a0f86'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 196, 'prompt_tokens': 6751, 'total_tokens': 6947, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 70, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 768}, 'prompt_cache_hit_tokens': 768, 'prompt_cache_miss_tokens': 5983}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'a323951b-0e69-4d16-a2a7-fe816f4c809d', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019edf21-46ee-79e2-9b6e-8906b

In [21]:
print(result["files"].keys())


dict_keys(['/workspace/langgraph.txt'])


In [27]:
print(result["files"]["/workspace/langgraph.txt"]["content"])

                     AI AGENTIC FRAMEWORKS & LANGGRAPH GUIDE
                      Compiled: 2025 (Latest Research)


PART 1: TOP AI AGENTIC FRAMEWORKS IN 2025

1. LANGCHAIN / LANGGRAPH
   - Most widely used agentic AI framework as of 2025.
   - Graph-based state machine approach for building complex, stateful agents.
   - Supports multi-agent orchestration, tool calling, memory, and conditional logic.
   - Steep learning curve but offers the most fine-grained control.
   - Best for: production-grade, complex agent workflows needing custom logic.

2. CREWAI
   - Role-based multi-agent framework where agents have specific roles, goals, and backstories.
   - Excellent for simulating teams of agents working together.
   - Latest version (0.95+) includes Anthropic/Google tool-call routing and async runners.
   - Best for: multi-agent collaboration scenarios.

3. MICROSOFT AUTOGEN
   - Open-source framework for building multi-agent conversations.
   - V2 introduced event-driven architecture